# Results Analysis

This notebook derives the agreement statistic and the final golden labels from the retained CSV files, then explains its results. The packets carry the narratives, so the core analysis needs no source extract; only section 5 (source labels versus golden labels) reads `ict3113_tickets.csv` when it is present, and it falls back to the retained `artifacts/source_vs_gold.csv` when it is not. Run all cells from the repository root or the `golden_set` folder. Output is aggregate counts and row IDs only.

Files read: `ernest-packet.csv` and `yp-packet.csv` (the independent label sheets), `artifacts/sampling_manifest.csv` (the 175 selected IDs), `resolution.csv` (decisions on reviewed cases) and `gold_labels.csv` (the golden test set). It writes `artifacts/review_list.csv`, a reference copy of the review cases for the reviewers, and `artifacts/source_vs_gold.csv`, the source label beside the golden label for each selected ID. The protocol is in `protocol.md`; the draw is in `selection_and_packets.ipynb`.

## 0. How the labels were produced

1. **Independent labelling.** Ernest and YP each filled their own packet using `protocol.md`: one of seven labels for the principal problem, an `uncertain` flag, and a reason for every `yes`. They did not discuss individual tickets, consult each other, or use the noisy source label.
2. **Preserve and validate.** The completed sheets and the resolution and final-label files were filled in by hand in Excel and imported here as CSV, so this notebook checks them rather than assuming they are well formed. Both completed sheets were checked (175 unique IDs in manifest order, matching narratives, allowed labels, a reason for every `yes` and none for `no`) and then left unchanged.
3. **Original agreement.** Exact agreement and nominal Cohen's kappa were calculated from the original labels before any case was discussed (section 2).
4. **Resolution.** Every disagreement and every row either rater marked uncertain was discussed and recorded in `resolution.csv`. A rule gap found in discussion produced a dated revision in `protocol.md`, applied to the review cases it touched. Rows on which both raters matched with no uncertainty were not reopened.
5. **Final set.** For each ID the final label is the `resolution.csv` decision if the row was reviewed, otherwise the matching labels from both packets are taken. (section 4).

## 1. Load and validate the retained records

In [1]:
from collections import Counter
import csv
from pathlib import Path
import re

RUN = Path.cwd()
if not (RUN / "gold_labels.csv").is_file():
    RUN = Path("golden_set")
assert (RUN / "gold_labels.csv").is_file(), "Run from the repository root or golden_set folder"
ARTIFACTS = RUN / "artifacts"

LABELS = (
    "Credit reporting", "Debt collection", "Mortgage", "Credit card",
    "Bank account or service", "Consumer loan", "Money transfer or service",
)

def read_csv(path, fields):
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        assert tuple(reader.fieldnames or ()) == fields, f"Unexpected columns: {path.name}"
        rows = list(reader)
    assert all(None not in row and all(v is not None for v in row.values()) for row in rows)
    return rows

packet_fields = ("row_id", "narrative", "label", "uncertain", "reason_if_uncertain")
manifest = read_csv(ARTIFACTS / "sampling_manifest.csv", ("row_id",))
ids = [row["row_id"] for row in manifest]
assert len(ids) == len(set(ids)) == 175

sheets = {
    "Ernest": read_csv(RUN / "ernest-packet.csv", packet_fields),
    "YP": read_csv(RUN / "yp-packet.csv", packet_fields),
}
for name, rows in sheets.items():
    assert [row["row_id"] for row in rows] == ids, f"{name}: IDs or order differ from the manifest"
    for row in rows:
        assert row["label"] in LABELS and row["uncertain"] in ("yes", "no")
        reason = row["reason_if_uncertain"].strip()
        assert (len(reason) >= 35) if row["uncertain"] == "yes" else not reason
assert [a["narrative"] for a in sheets["Ernest"]] == [b["narrative"] for b in sheets["YP"]]

ernest = {row["row_id"]: row for row in sheets["Ernest"]}
yp = {row["row_id"]: row for row in sheets["YP"]}
print("Validated 175 manifest IDs and one complete rating per rater for each.")

Validated 175 manifest IDs and one complete rating per rater for each.


## 2. Original agreement

Only the two `label` columns of the unchanged original sheets are used, before any resolution. The seven-by-seven contingency table counts how often Ernest chose the row label while YP chose the column label.

- Observed agreement: `p_o = exact matches / N`.
- Chance-expected agreement: `p_e = sum over labels of (Ernest total x YP total) / N^2`, the agreement expected if each rater chose labels independently at their own observed rates.
- Nominal Cohen's kappa: `kappa = (p_o - p_e) / (1 - p_e)`, the share of the possible above-chance agreement that was achieved. Labels are unordered categories, so every mismatch counts as equally wrong.

In [2]:
n = len(ids)
matrix = Counter((ernest[i]["label"], yp[i]["label"]) for i in ids)
row_totals = Counter(ernest[i]["label"] for i in ids)
col_totals = Counter(yp[i]["label"] for i in ids)

matches = sum(matrix[label, label] for label in LABELS)
p_o = matches / n
p_e = sum(row_totals[label] * col_totals[label] for label in LABELS) / n**2
kappa = (p_o - p_e) / (1 - p_e)

assert matches == 165 and f"{p_o:.6f}" == "0.942857"
assert f"{p_e:.6f}" == "0.149420" and f"{kappa:.6f}" == "0.932819"

print(f"Exact matches: {matches}/{n} = {p_o:.6f}")
print(f"Chance-expected agreement: sum(Ernest total x YP total) / {n}^2 = {p_e:.6f}")
print(f"Nominal Cohen's kappa: (p_o - p_e) / (1 - p_e) = {kappa:.6f}")

short = ["CR", "DC", "Mort", "Card", "Bank", "Loan", "Xfer"]
print("\nContingency table (rows Ernest, columns YP):")
print(f"{'':28}" + "".join(f"{s:>6}" for s in short) + f"{'Total':>7}")
for label in LABELS:
    print(f"{label:28}" + "".join(f"{matrix[label, other]:>6}" for other in LABELS) + f"{row_totals[label]:>7}")
print(f"{'Total':28}" + "".join(f"{col_totals[other]:>6}" for other in LABELS) + f"{n:>7}")

print("\nDisagreements (Ernest -> YP):")
disagreements = {pair: c for pair, c in matrix.items() if pair[0] != pair[1]}
for (a, b), c in sorted(disagreements.items(), key=lambda kv: -kv[1]):
    print(f"  {c}  {a} -> {b}")
assert sum(disagreements.values()) == n - matches == 10

Exact matches: 165/175 = 0.942857
Chance-expected agreement: sum(Ernest total x YP total) / 175^2 = 0.149420
Nominal Cohen's kappa: (p_o - p_e) / (1 - p_e) = 0.932819

Contingency table (rows Ernest, columns YP):
                                CR    DC  Mort  Card  Bank  Loan  Xfer  Total
Credit reporting                35     0     0     0     0     0     0     35
Debt collection                  2    23     0     0     0     0     0     25
Mortgage                         0     0    22     0     0     0     0     22
Credit card                      0     0     0    20     1     1     0     22
Bank account or service          0     0     0     0    26     0     4     30
Consumer loan                    1     0     0     0     0    22     0     23
Money transfer or service        0     0     0     0     1     0    17     18
Total                           38    23    22    20    28    23    21    175

Disagreements (Ernest -> YP):
  4  Bank account or service -> Money transfer or serv

## 3. Review cases

The review list contains every disagreement **and** every row either rater marked uncertain, so a row with matching labels is still reviewed if one of them was unsure. It is derived here from the two sheets and written to `artifacts/review_list.csv`: each review case with both raters' labels, uncertainty flags and reasons side by side. This file is only for the reviewers' ease of reference while they discuss the cases; nothing later in the notebook reads it, and the final labels come from `resolution.csv` and the packets.

In [3]:
triggers = {}
for i in ids:
    reasons = []
    if ernest[i]["label"] != yp[i]["label"]:
        reasons.append("disagreement")
    if ernest[i]["uncertain"] == "yes" or yp[i]["uncertain"] == "yes":
        reasons.append("uncertain")
    if reasons:
        triggers[i] = ";".join(reasons)

REVIEW_FIELDS = (
    "row_id", "narrative", "ernest_label", "ernest_uncertain", "ernest_reason_if_uncertain",
    "yp_label", "yp_uncertain", "yp_reason_if_uncertain", "review_trigger")
review_rows = [
    {"row_id": i, "narrative": ernest[i]["narrative"],
     "ernest_label": ernest[i]["label"], "ernest_uncertain": ernest[i]["uncertain"],
     "ernest_reason_if_uncertain": ernest[i]["reason_if_uncertain"],
     "yp_label": yp[i]["label"], "yp_uncertain": yp[i]["uncertain"],
     "yp_reason_if_uncertain": yp[i]["reason_if_uncertain"], "review_trigger": triggers[i]}
    for i in ids if i in triggers
]
with (ARTIFACTS / "review_list.csv").open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=REVIEW_FIELDS, lineterminator="\n")
    writer.writeheader()
    writer.writerows(review_rows)
assert len(triggers) == len(review_rows) == 51

by_trigger = Counter(triggers.values())
print("Review cases:", len(triggers))
print("  uncertain only:", by_trigger["uncertain"])
print("  disagreement only:", by_trigger["disagreement"])
print("  disagreement and uncertain:", by_trigger["disagreement;uncertain"])
print("Uncertainty flags: Ernest", sum(ernest[i]["uncertain"] == "yes" for i in ids),
      "| YP", sum(yp[i]["uncertain"] == "yes" for i in ids),
      "| both", sum(ernest[i]["uncertain"] == yp[i]["uncertain"] == "yes" for i in ids))

Review cases: 51
  uncertain only: 41
  disagreement only: 0
  disagreement and uncertain: 10
Uncertainty flags: Ernest 46 | YP 35 | both 30


## 4. Resolution and the final set

`resolution.csv` holds one decision per reviewed row: the narrative, the final explanation, and the final label. It covers exactly the 51 review cases: every disagreement and every row either rater marked uncertain. The golden labels are then rebuilt from the rules in section 0 and compared with `gold_labels.csv`.

In [4]:
resolution = read_csv(RUN / "resolution.csv", ("row_id", "narrative", "final_explanation", "final_label"))
decision = {row["row_id"]: row for row in resolution}
assert len(decision) == len(resolution) == 51
assert set(decision) == set(triggers), "resolution.csv must cover exactly the review cases"
for i, row in decision.items():
    assert row["narrative"] == ernest[i]["narrative"]
    assert row["final_label"] in LABELS and len(row["final_explanation"].strip()) >= 30

gold = read_csv(RUN / "gold_labels.csv", ("row", "gold_label"))
assert [row["row"] for row in gold] == ids
carried = 0
for row in gold:
    i = row["row"]
    if i in decision:
        expected = decision[i]["final_label"]
    else:
        assert ernest[i]["label"] == yp[i]["label"] and ernest[i]["uncertain"] == yp[i]["uncertain"] == "no"
        expected = ernest[i]["label"]
        carried += 1
    assert row["gold_label"] == expected
counts = Counter(row["gold_label"] for row in gold)
assert len(gold) == 175 and carried == 175 - 51 == 124

print("Resolved cases:", len(decision), "= review cases", len(triggers))
print("Certain matching originals carried forward:", carried)
changed = sum(decision[i]["final_label"] not in (ernest[i]["label"], yp[i]["label"]) for i in decision)
print("Reviewed rows where the final label differs from both originals:", changed)
print("\nGolden test set label counts:")
for label in LABELS:
    print(f"  {label}: {counts[label]}")
print("  Total:", sum(counts.values()))

Resolved cases: 51 = review cases 51
Certain matching originals carried forward: 124
Reviewed rows where the final label differs from both originals: 2

Golden test set label counts:
  Credit reporting: 36
  Debt collection: 24
  Mortgage: 22
  Credit card: 21
  Bank account or service: 33
  Consumer loan: 23
  Money transfer or service: 16
  Total: 175


## 5. Source labels versus the golden labels

The draw was stratified on the raw `source_label` (25 per category), but that label was never shown to the raters and is not an answer key. Now that the golden labels are final, comparing the two shows what the raw labels are worth as ground truth, which is the reason a golden set was built at all.

Each selected ID is paired with its source label and its golden label, and the pairs are saved to `artifacts/source_vs_gold.csv` so the comparison survives without the large extract. The table has source category down the side and golden category across the top; the diagonal is where the two agree. Because exactly 25 tickets were drawn per source category, every row total is 25 by construction. Read the off-diagonal cells as the direction of the raw-label noise, not as a population error rate: this sample is balanced on the noisy label, not on the real category mix.

In [5]:
SOURCE = RUN / "ict3113_tickets.csv"
PAIRS = ARTIFACTS / "source_vs_gold.csv"
PAIR_FIELDS = ("row_id", "source_label", "gold_label")
gold_of = {row["row"]: row["gold_label"] for row in gold}

if SOURCE.is_file():
    csv.field_size_limit(10**8)
    wanted = set(ids)
    with SOURCE.open("r", encoding="utf-8-sig", newline="") as stream:
        source_of = {r["row"]: r["source_label"] for r in csv.DictReader(stream) if r["row"] in wanted}
    assert set(source_of) == wanted
    pair_rows = [{"row_id": i, "source_label": source_of[i], "gold_label": gold_of[i]} for i in ids]
    with PAIRS.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=PAIR_FIELDS, lineterminator="\n")
        writer.writeheader()
        writer.writerows(pair_rows)
    origin = "the source extract"
elif PAIRS.is_file():
    pair_rows = read_csv(PAIRS, PAIR_FIELDS)
    origin = "the retained artifacts/source_vs_gold.csv"
else:
    pair_rows = None

if pair_rows is None:
    print("Skipped: neither ict3113_tickets.csv nor artifacts/source_vs_gold.csv is available.")
else:
    assert [r["row_id"] for r in pair_rows] == ids
    assert all(gold_of[r["row_id"]] == r["gold_label"] and r["source_label"] in LABELS for r in pair_rows)
    table = Counter((r["source_label"], r["gold_label"]) for r in pair_rows)
    same = sum(table[label, label] for label in LABELS)
    print(f"Source read from {origin}.")
    print(f"Source label equals golden label: {same}/{n} = {same / n:.1%}; differs: {n - same} ({(n - same) / n:.1%})")
    print("\nRows source category, columns golden category:")
    print(f"{'':28}" + "".join(f"{s:>6}" for s in short) + f"{'Total':>7}")
    for label in LABELS:
        print(f"{label:28}" + "".join(f"{table[label, other]:>6}" for other in LABELS)
              + f"{sum(table[label, other] for other in LABELS):>7}")
    print(f"{'Total':28}" + "".join(f"{sum(table[other, label] for other in LABELS):>6}" for label in LABELS) + f"{n:>7}")
    print("\nHow much of each source category the golden label kept:")
    for label in LABELS:
        print(f"  {label}: {table[label, label]}/25 kept")
    print("\nLargest source -> golden moves:")
    moves = sorted(((c, a, b) for (a, b), c in table.items() if a != b), reverse=True)
    for c, a, b in moves[:5]:
        print(f"  {c}  {a} -> {b}")

Source read from the source extract.
Source label equals golden label: 132/175 = 75.4%; differs: 43 (24.6%)

Rows source category, columns golden category:
                                CR    DC  Mort  Card  Bank  Loan  Xfer  Total
Credit reporting                21     2     0     0     1     1     0     25
Debt collection                  5    20     0     0     0     0     0     25
Mortgage                         0     1    22     0     1     1     0     25
Credit card                      6     0     0    16     2     0     1     25
Bank account or service          0     0     0     3    20     0     2     25
Consumer loan                    4     1     0     0     0    20     0     25
Money transfer or service        0     0     0     2     9     1    13     25
Total                           36    24    22    21    33    23    16    175

How much of each source category the golden label kept:
  Credit reporting: 21/25 kept
  Debt collection: 20/25 kept
  Mortgage: 22/25 kept
 

## 6. Why agreement was high

The numbers below back the discussion that follows them.

In [6]:
flagged = [i for i in ids if ernest[i]["label"] != yp[i]["label"]]
flagged_by_someone = [i for i in flagged if "yes" in (ernest[i]["uncertain"], yp[i]["uncertain"])]
clear = [i for i in ids if ernest[i]["uncertain"] == yp[i]["uncertain"] == "no"]
clear_match = [i for i in clear if ernest[i]["label"] == yp[i]["label"]]
either = [i for i in ids if "yes" in (ernest[i]["uncertain"], yp[i]["uncertain"])]
either_match = [i for i in either if ernest[i]["label"] == yp[i]["label"]]

print(f"Rows neither rater flagged uncertain: {len(clear)}; matching labels: {len(clear_match)} ({len(clear_match)/len(clear):.1%})")
print(f"Rows either rater flagged uncertain: {len(either)}; matching labels: {len(either_match)} ({len(either_match)/len(either):.1%})")
print(f"Disagreements where at least one rater had flagged uncertainty: {len(flagged_by_someone)}/{len(flagged)}")

bank_xfer = sum(c for (a, b), c in matrix.items() if {a, b} == {"Bank account or service", "Money transfer or service"})
print(f"Disagreements between Bank account or service and Money transfer or service: {bank_xfer}/{len(flagged)}")

print(f"\nChance-expected agreement with these marginals: {p_e:.3f}; with seven equally likely labels: {1/7:.3f}")
print("Agreement per Ernest label (matches / Ernest total):")
for label in LABELS:
    print(f"  {label}: {matrix[label, label]}/{row_totals[label]}")

Rows neither rater flagged uncertain: 124; matching labels: 124 (100.0%)
Rows either rater flagged uncertain: 51; matching labels: 41 (80.4%)
Disagreements where at least one rater had flagged uncertainty: 10/10
Disagreements between Bank account or service and Money transfer or service: 5/10

Chance-expected agreement with these marginals: 0.149; with seven equally likely labels: 0.143
Agreement per Ernest label (matches / Ernest total):
  Credit reporting: 35/35
  Debt collection: 23/25
  Mortgage: 22/22
  Credit card: 20/22
  Bank account or service: 26/30
  Consumer loan: 22/23
  Money transfer or service: 17/18


### Factors that likely raised agreement

- **Prior discussion of the protocol.** The group discussed and agreed on the label set, the decision rules and how to use the uncertainty flag before this draw. That shared calibration on what each category means, and on how to treat overlaps is the largest single reason the raters converged. It also means the agreement measures how well the *pair* applies a shared protocol, not how well strangers would.
- **Well-separated categories and a balanced draw.** The seven labels are distinct financial products or problems, and the draw took 25 tickets from each source stratum. Chance-expected agreement is therefore low (0.149, close to the 1/7 = 0.143 of seven equally likely labels), so the high raw agreement translates into a high kappa (0.933) rather than being discounted heavily.
- **An uncertainty flag that captured hard cases.** Every ticket that was a genuine judgement call could be marked `uncertain` with a reason. The rows where neither rater expressed doubt were labelled identically (124 of 124), and all 10 disagreements involved a row at least one rater had flagged (see the counts above), so disagreement was concentrated in cases both raters already knew were borderline.

### Where the protocol was weakest, and what it changed

Bank account versus money transfer accounts for the largest share of the disagreements. The protocol did not say what to do with payment-platform complaints about suspended accounts, identity-verification locks, or held balances. Discussion of those cases produced the dated 29 September 2026 revision (account access with no specific transfer at issue is `Bank account or service`; a specific failed, disputed or unrecovered transfer is `Money transfer or service`). 

### Limits of the statistic

- Agreement measures consistency between two raters, not correctness. A shared misreading of a category would raise agreement without making the labels right.
- The same pair helped shape the protocol and then applied it, which favours agreement relative to an outside annotator.
- The source labels used to stratify the draw are noisy and were not used as answers. Final label counts (above) differ from the 25-per-stratum draw, so any later accuracy figure describes performance on this test set, not on the whole extract's category mix.
- The source-label comparison (section 5) is on a sample balanced on the noisy label, so it shows the direction of raw-label noise, not a population error rate.### Factors that likely raised agreement

- **Prior discussion of the protocol.** The group discussed and agreed on the label set, the decision rules and how to use the uncertainty flag before this draw. That shared calibration on what each category means, and on how to treat overlaps is the largest single reason the raters converged. It also means the agreement measures how well the *pair* applies a shared protocol, not how well strangers would.
- **Well-separated categories and a balanced draw.** The seven labels are distinct financial products or problems, and the draw took 25 tickets from each source stratum. Chance-expected agreement is therefore low (0.149, close to the 1/7 = 0.143 of seven equally likely labels), so the high raw agreement translates into a high kappa (0.933) rather than being discounted heavily.
- **An uncertainty flag that captured hard cases.** Every ticket that was a genuine judgement call could be marked `uncertain` with a reason. The rows where neither rater expressed doubt were labelled identically (124 of 124), and all 10 disagreements involved a row at least one rater had flagged (see the counts above), so disagreement was concentrated in cases both raters already knew were borderline.

### Where the protocol was weakest, and what it changed

Bank account versus money transfer accounts for the largest share of the disagreements. An Addendum was made because the protocol did not say what to do with payment-platform complaints about suspended accounts, identity-verification locks, or held balances. Discussion of those cases produced the dated 29 September 2026 revision (account access with no specific transfer at issue is `Bank account or service`; a specific failed, disputed or unrecovered transfer is `Money transfer or service`). 

### Limits of the statistic

- Agreement measures consistency between two raters, not correctness. A shared misreading of a category would raise agreement without making the labels right.
- The same pair helped shape the protocol and then applied it, which favours agreement relative to an outside annotator.
- The source labels used to stratify the draw are noisy and were not used as answers. Final label counts (above) differ from the 25-per-stratum draw, so any later accuracy figure describes performance on this test set, not on the whole extract's category mix.
- The source-label comparison (section 5) is on a sample balanced on the noisy label, so it shows the direction of raw-label noise, not a population error rate.